# 20 — Pseudolive: regla congelada sobre datos no usados

> **Estado: ESQUELETO.** Este notebook fija el diseño antes de implementar nada. Todas las secciones de trabajo son celdas `TODO` controladas por el flag `IMPLEMENTADO` (por defecto `False`): con `False` el notebook se ejecuta de principio a fin sin error y solo informa de lo pendiente; con `True` cada `TODO` sin implementar levanta `NotImplementedError`. Ninguna cifra de este texto está escrita a mano: las que aparezcan saldrán de celdas.

## 0. Objetivo

Validar la tool/señal de régimen de la Fase 1 del TFM (`get_regimen(fecha)`, [ADR-005](../docs/decisions/ADR-005-reencuadre-tfm-multiagente.md)) sobre datos no usados.

Aplicar la regla congelada en `19` **sin tocarla** a (a) un bloque de datos no usado en ninguna
fase anterior y (b) los datos que vayan llegando después (seguimiento hacia delante), y
contrastar el resultado con el criterio de éxito pre-registrado en la futura ADR-006 (la decisión de `19`). La salida que se valida es la de la Fase 1: la señal que expondrá `get_regimen(fecha)` a los agentes (estado, probabilidad, días en el estado).

## Preguntas que responde

1. ¿La regla congelada se comporta en el bloque no usado como anticipaba el benchmark
   (estabilidad, tasa de alarmas, estado en episodios de estrés)?
2. ¿Se sostiene hacia delante, con los datos tal y como se publican (lags de publicación)?
3. ¿Cumple o incumple el criterio de éxito/fracaso de ADR-006?

## Entradas y salidas (vía `regimenes.rutas`)

| papel | ruta |
|---|---|
| regla congelada | `rutas.RESULTS_PSEUDOLIVE / 'regla_congelada.json'` |
| datos crudos (incluida la cola posterior a la ventana del benchmark) | `rutas.DATA_RAW` |
| catálogo y banco de pruebas | `rutas.CATALOG`, `rutas.BENCHMARK_SPEC` |
| diario de estados (solo añadir) | `rutas.RESULTS_PSEUDOLIVE / 'diario.csv'` |
| evaluación | `rutas.RESULTS_PSEUDOLIVE / 'evaluacion'` |

## Metodología prevista

- **Verificación de congelación**: se comprueba la huella de código y de parámetros guardada en
  la regla; si no coincide, el notebook falla (no se evalúa una regla distinta de la decidida).
- **Bloque no usado**: sesiones posteriores a `ventana_fin` de `configs/benchmark_spec.yaml`
  (la celda de configuración imprime la fecha) disponibles en `data/raw`, más cualquier bloque
  apartado explícitamente en ADR-006. Features con `regimenes.features` (mismos lags de
  publicación) y detectores ajustados solo con datos hasta el corte.
- **Seguimiento hacia delante**: actualización con `python -m regimenes.datos`, reconstrucción
  causal y **anotación en un diario solo-añadir** del estado emitido, la fecha de cálculo y la
  vintage de datos usada; nunca se reescriben filas pasadas.
- **Evaluación**: métricas del benchmark sobre el bloque y lectura cualitativa de episodios;
  con muestra corta se informa de la potencia (probable ausencia de crisis) antes de concluir.

## Riesgos

- **Re-ajuste encubierto.** Cualquier cambio de regla tras ver el bloque invalida el pseudolive:
  se registra como nueva versión con nueva fecha de congelación.
- **Revisiones de datos.** La historia es la última vintage (no tiempo real); el diario guarda
  la vintage usada en cada fecha para separar señal de revisión.
- **Muestra escasa.** Puede no haber ninguna crisis en el bloque; el resultado sería no
  concluyente, no un éxito.
- **Fuga.** El bloque no usado no puede haber entrado en `02`–`19` (se verifica por fechas).

**Índice**

0. Objetivo, preguntas, entradas/salidas, metodología y riesgos
1. Configuración
2. Carga y verificación de la regla congelada
3. Bloque no usado
4. Seguimiento hacia delante
5. Evaluación frente al criterio pre-registrado


## 1. Configuración

In [1]:
# Configuración común del esqueleto (no lee results/benchmark ni ajusta nada).
import yaml
import pandas as pd
from IPython.display import display

from regimenes import rutas
from regimenes import sinteticos
from regimenes.sinteticos import Generador
from regimenes.sinteticos import registry as registro_sinteticos

IMPLEMENTADO = False  # pasar a True solo cuando las secciones TODO estén implementadas

CONFIG_SINTETICOS = yaml.safe_load(rutas.SINTETICOS_CONFIG.read_text(encoding='utf-8'))
SEMILLA = CONFIG_SINTETICOS.get('semilla')


def saltar(seccion: str, detalle: str) -> None:
    """Punto TODO controlado: informa y sigue (solo se llama con IMPLEMENTADO=False)."""
    print(f'[TODO pendiente] {seccion}: {detalle}')


def tabla_rutas(rutas_nb: dict) -> pd.DataFrame:
    """Entradas/salidas del notebook (siempre vía regimenes.rutas) y si existen ya en disco."""
    filas = [
        {'papel': papel, 'ruta': ruta.relative_to(rutas.ROOT).as_posix(), 'existe': ruta.exists()}
        for papel, ruta in rutas_nb.items()
    ]
    return pd.DataFrame(filas).set_index('papel')
from regimenes.benchmark import load_benchmark_spec

RUTAS_NB = {
    'entrada: regla congelada': rutas.RESULTS_PSEUDOLIVE / 'regla_congelada.json',
    'entrada: datos crudos': rutas.DATA_RAW,
    'entrada: catálogo': rutas.CATALOG,
    'entrada: banco de pruebas': rutas.BENCHMARK_SPEC,
    'salida: diario (solo añadir)': rutas.RESULTS_PSEUDOLIVE / 'diario.csv',
    'salida: evaluación': rutas.RESULTS_PSEUDOLIVE / 'evaluacion',
}
display(tabla_rutas(RUTAS_NB))
SPEC = load_benchmark_spec()
FIN_BENCHMARK = {track: SPEC[f'pista_{track}']['ventana_fin'] for track in ('A', 'B')}
print('Fin de ventana del benchmark por pista (el bloque no usado empieza después):', FIN_BENCHMARK)


,ruta,existe
papel,,
entrada: regla congelada,results/pseudolive/regla_congelada.json,False
entrada: datos crudos,data/raw,True
entrada: catálogo,configs/catalog.yaml,True
entrada: banco de pruebas,configs/benchmark_spec.yaml,True
salida: diario (solo añadir),results/pseudolive/diario.csv,False
salida: evaluación,results/pseudolive/evaluacion,False


Fin de ventana del benchmark por pista (el bloque no usado empieza después): {'A': '2026-05-29', 'B': '2026-05-29'}


## 2. Carga y verificación de la regla congelada

In [2]:
# TODO 20.2 — verificar la regla congelada
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. leer rutas.RESULTS_PSEUDOLIVE / 'regla_congelada.json' (escrita en 19)
    # 2. comparar huella de código/parámetros con la actual; fallar si difiere
    raise NotImplementedError('TODO 20.2: cargar y verificar la regla congelada')
else:
    saltar('20.2', 'verificar la regla congelada')


[TODO pendiente] 20.2: verificar la regla congelada


## 3. Bloque no usado

In [3]:
# TODO 20.3 — aplicar la regla al bloque no usado
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes import features as ft

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. construir features causales tras FIN_BENCHMARK con regimenes.features (mismos lags)
    # 2. detectores ajustados solo hasta el corte; fusión con la FusionConfig congelada
    # 3. verificar por fechas que ninguna sesión del bloque entró en 02-19
    raise NotImplementedError('TODO 20.3: evaluar la regla en el bloque no usado')
else:
    saltar('20.3', 'aplicar la regla al bloque no usado')


[TODO pendiente] 20.3: aplicar la regla al bloque no usado


## 4. Seguimiento hacia delante

In [4]:
# TODO 20.4 — diario hacia delante
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. tras `python -m regimenes.datos`, recalcular causalmente el último estado
    # 2. añadir fila (fecha_dato, fecha_calculo, estado, vintage) a rutas.RESULTS_PSEUDOLIVE / 'diario.csv' sin reescribir
    raise NotImplementedError('TODO 20.4: mantener el diario de seguimiento hacia delante')
else:
    saltar('20.4', 'diario hacia delante')


[TODO pendiente] 20.4: diario hacia delante


## 5. Evaluación frente al criterio pre-registrado

In [5]:
# TODO 20.5 — evaluación frente a ADR-006
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. métricas del benchmark sobre el bloque + potencia; veredicto éxito/fracaso/no concluyente según ADR-006
    raise NotImplementedError('TODO 20.5: evaluar frente al criterio pre-registrado')
else:
    saltar('20.5', 'evaluación frente a ADR-006')


[TODO pendiente] 20.5: evaluación frente a ADR-006
